In [ ]:
import glob, zipfile, os
import subprocess, sys

INPUT_DIR = "/kaggle/input/datasets/sabbir4724/li-ion-battery-dataset-from-nasa-pcoe"
OUTPUT_ROOT = "/kaggle/working/"
EXTRACT_DIR = "/kaggle/working/nasa_data"

if not glob.glob(os.path.join(INPUT_DIR, "**", "*.mat"), recursive=True):
    if not glob.glob(os.path.join(EXTRACT_DIR, "**", "*.mat"), recursive=True):
        zips = glob.glob(os.path.join(INPUT_DIR, "*.zip"))
        if not zips:
            zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if not zips:
            raise FileNotFoundError(f"No .mat or .zip found in {INPUT_DIR}")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(EXTRACT_DIR)
        print(f"Extracted {zips[0]} -> {EXTRACT_DIR}")
        BASE_PATH = EXTRACT_DIR
    else:
        BASE_PATH = EXTRACT_DIR
else:
    BASE_PATH = INPUT_DIR
    print("Using raw .mat files from Kaggle Input.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"])

import os, re, glob, time, json, pickle, warnings, subprocess, sys, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io
from scipy import stats

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, RegressorMixin, clone

import xgboost as xgb

warnings.filterwarnings("ignore")

try:
    display
except NameError:
    display = print

try:
    import optuna
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

CODE_VERSION = "v2"
SEED = 42
ROBUST_SEEDS = [42, 7, 21, 0, 1]

DPI = 300
RUN_RANDOM_SPLIT = True
RUN_CHRONO_SPLIT = False         
RUN_ABLATION = True
RUN_CHANNEL_ABLATION = True
RUN_ROBUSTNESS = True
RUN_UNCERTAINTY = True
PLOT_TRAINING_CURVES = True        
PLOT_BASELINE_CURVES = False       
PLOT_ALL_MODEL_CURVES = True

SEQ_LEN = 300
RATED_CAPACITY = 2.0
EOL_SOH_THRESHOLD = 70.0

XGB_SCALES = (1, 2, 4, 8)
N_BASE = 2 * len(XGB_SCALES)
BASE_NAMES = [f"XGB(seg={s})" for s in XGB_SCALES] + [f"MS-SVR(seg={s})" for s in XGB_SCALES]

TARGET_BATTERIES = ["B0005", "B0006", "B0007", "B0018"]
TARGETS = ["SOH", "RUL"]

PROPOSED = "Proposed (MS-XGB + MS-SVR + BiGRU-Attn)"

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = OUTPUT_ROOT
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG_SIG = hashlib.md5(
    json.dumps(
        {
            "code": CODE_VERSION,
            "seed": SEED,
            "scales": XGB_SCALES,
            "seq_len": SEQ_LEN,
            "rated": RATED_CAPACITY,
            "eol": EOL_SOH_THRESHOLD,
            "batteries": TARGET_BATTERIES,
        },
        sort_keys=True,
    ).encode()
).hexdigest()[:8]

CACHE_DIR = os.path.join(OUTPUT_DIR, f"cache_{CONFIG_SIG}")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Cache directory : {CACHE_DIR}")

def out(fname):
    return os.path.join(OUTPUT_DIR, fname)

def slug(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)

def wpath(tag):
    return os.path.join(CACHE_DIR, slug(tag) + ".weights.h5")

def cached(key, fn):
    path = os.path.join(CACHE_DIR, slug(key) + ".pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    result = fn()
    with open(path, "wb") as f:
        pickle.dump(result, f)
    return result

def neural_tag(bundle, tag):
    return f"{tag}_{bundle['sig']}"

mat_files = glob.glob(os.path.join(BASE_PATH, "**", "*.mat"), recursive=True)

if len(mat_files) == 0:
    raise FileNotFoundError(f"No .mat files found under {BASE_PATH}.")

print(f"Found {len(mat_files)} .mat files in {BASE_PATH}")

def load_battery(mat_file, battery_id):
    mat = scipy.io.loadmat(mat_file)
    keys = [k for k in mat.keys() if not k.startswith("__")]
    if not keys:
        return None
    cycles = mat[keys[0]][0][0]["cycle"][0]
    X_seq, y_soh, cycle_indices = [], [], []
    discharge_cycle = 0
    for cycle in cycles:
        if cycle["type"][0] != "discharge":
            continue
        discharge_cycle += 1
        data = cycle["data"][0][0]
        V = data["Voltage_measured"][0].astype(float)
        I = data["Current_measured"][0].astype(float)
        T = data["Temperature_measured"][0].astype(float)
        t = data["Time"][0].astype(float)
        if len(t) < 2:
            continue
        t_norm = np.linspace(0, 1, len(t))
        t_target = np.linspace(0, 1, SEQ_LEN)
        V_i = np.interp(t_target, t_norm, V)
        I_i = np.interp(t_target, t_norm, I)
        T_i = np.interp(t_target, t_norm, T)
        X_seq.append(np.stack([V_i, I_i, T_i], axis=-1))
        capacity = data["Capacity"][0][0] if "Capacity" in data.dtype.names else np.nan
        y_soh.append((capacity / RATED_CAPACITY) * 100.0)
        cycle_indices.append(discharge_cycle)
    X_seq = np.asarray(X_seq, dtype=float)
    y_soh = np.asarray(y_soh, dtype=float)
    cycle_indices = np.asarray(cycle_indices)
    eol_positions = np.where(y_soh <= EOL_SOH_THRESHOLD)[0]
    if len(eol_positions) == 0:
        y_rul = np.full(len(y_soh), np.nan)
        eol_reached = False
    else:
        eol_cycle = cycle_indices[eol_positions[0]]
        y_rul = np.clip(eol_cycle - cycle_indices, 0, None)
        eol_reached = True
    return {
        "battery": battery_id, "X": X_seq, "SOH": y_soh, "RUL": y_rul,
        "cycle": cycle_indices, "eol_reached": eol_reached,
    }

X_all, y_soh_all, y_rul_all = [], [], []
bat_ids_all, cyc_all, eol_flags = [], [], []

for battery in TARGET_BATTERIES:
    path = next((f for f in mat_files if battery in f), None)
    if path is None:
        print(f"Warning: {battery} not found.")
        continue
    result = load_battery(path, battery)
    if result is None:
        continue
    X_all.append(result["X"])
    y_soh_all.append(result["SOH"])
    y_rul_all.append(result["RUL"])
    bat_ids_all.append(np.array([battery] * len(result["X"])))
    cyc_all.append(result["cycle"])
    eol_flags.append(np.array([result["eol_reached"]] * len(result["X"])))
    print(f"Loaded {battery}: {result['X'].shape[0]} cycles, EOL reached = {result['eol_reached']}")

X_all = np.concatenate(X_all)
y_soh_all = np.concatenate(y_soh_all)
y_rul_all = np.concatenate(y_rul_all)
bat_ids_all = np.concatenate(bat_ids_all)
cyc_all = np.concatenate(cyc_all)
eol_flags = np.concatenate(eol_flags)

BATS_L = [b for b in TARGET_BATTERIES if (bat_ids_all == b).any()]
RUL_VALID_MASK = eol_flags & ~np.isnan(y_rul_all)

print(f"Total cycles: {len(X_all)} | valid for RUL: {int(RUL_VALID_MASK.sum())}")

def y_of(target):
    return y_soh_all if target == "SOH" else y_rul_all

def valid_mask_of(target):
    if target == "SOH":
        return np.ones(len(X_all), dtype=bool)
    return RUL_VALID_MASK

def random_split(idx, seed=SEED):
    train, rest = train_test_split(idx, test_size=0.40, random_state=seed)
    val, rest = train_test_split(rest, test_size=0.625, random_state=seed)
    cal, test = train_test_split(rest, test_size=0.60, random_state=seed)
    return {"train": train, "val": val, "cal": cal, "test": test}

def chronological_split(idx):
    parts = {"train": [], "val": [], "cal": [], "test": []}
    for battery in BATS_L:
        battery_idx = idx[bat_ids_all[idx] == battery]
        if len(battery_idx) == 0:
            continue
        battery_idx = battery_idx[np.argsort(cyc_all[battery_idx])]
        n = len(battery_idx)
        n_train = int(round(0.60 * n))
        n_val = int(round(0.15 * n))
        n_cal = int(round(0.10 * n))
        parts["train"].append(battery_idx[:n_train])
        parts["val"].append(battery_idx[n_train:n_train + n_val])
        parts["cal"].append(battery_idx[n_train + n_val:n_train + n_val + n_cal])
        parts["test"].append(battery_idx[n_train + n_val + n_cal:])
    return {key: np.concatenate(value) for key, value in parts.items()}

def make_splits(target, mode):
    valid_idx = np.where(valid_mask_of(target))[0]
    if mode == "random":
        return random_split(valid_idx)
    if mode == "chrono":
        return chronological_split(valid_idx)
    raise ValueError(f"Unknown split mode: {mode}")

METRICS = ["RMSE", "MAE", "MAPE (%)", "sMAPE (%)", "R2"]

def get_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mask = np.abs(y_true) > 1.0
    if mask.any():
        mape = np.mean(np.abs(y_pred[mask] - y_true[mask]) / np.abs(y_true[mask])) * 100.0
    else:
        mape = np.nan
    smape = np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100.0
    return {"RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "sMAPE (%)": smape, "R2": r2_score(y_true, y_pred)}

STAT_FUNCS = {
    0: [np.max, np.min, np.mean, np.std],
    1: [np.max, np.min, np.mean],
    2: [np.max, np.min, np.mean, np.std],
}

def extract_stat_features(X_seq, channels=(0, 1, 2)):
    columns = []
    for channel in channels:
        for function in STAT_FUNCS[channel]:
            columns.append(function(X_seq[:, :, channel], axis=1))
    return np.column_stack(columns)

def extract_ms_features(X_seq, channels, n_segments):
    columns = []
    for channel in channels:
        segments = np.array_split(np.arange(X_seq.shape[1]), n_segments)
        for segment in segments:
            x = X_seq[:, segment, channel]
            for function in STAT_FUNCS[channel]:
                columns.append(function(x, axis=1))
            columns.append(x[:, -1] - x[:, 0])
    return np.column_stack(columns)

def scale_sequence(scaler, X):
    flat = X.reshape(-1, X.shape[-1])
    transformed = scaler.transform(flat)
    return transformed.reshape(X.shape)

class MultiScaleKernelSVR(BaseEstimator, RegressorMixin):
    def __init__(self, gammas=(0.01, 0.05, 0.2, 1.0), C=10.0, epsilon=0.01, weights=None, n_landmarks=300, seed=42):
        self.gammas = gammas
        self.C = C
        self.epsilon = epsilon
        self.weights = weights
        self.n_landmarks = n_landmarks
        self.seed = seed

    def _features(self, X, landmarks, weights):
        blocks = []
        for gamma, weight in zip(self.gammas, weights):
            kernel = rbf_kernel(X, landmarks, gamma=gamma)
            blocks.append(np.sqrt(max(weight, 1e-8)) * kernel)
        return np.hstack(blocks)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        rng = np.random.RandomState(self.seed)
        n_landmarks = min(self.n_landmarks, len(X))
        self.landmarks_ = X[rng.choice(len(X), n_landmarks, replace=False)]
        if self.weights is None:
            self.weights_ = np.ones(len(self.gammas)) / len(self.gammas)
        else:
            weights = np.clip(np.asarray(self.weights, dtype=float), 1e-6, None)
            self.weights_ = weights / weights.sum()
        Z = self._features(X, self.landmarks_, self.weights_)
        self.svr_ = SVR(C=self.C, epsilon=self.epsilon, kernel="rbf", gamma="scale").fit(Z, y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        Z = self._features(X, self.landmarks_, self.weights_)
        return self.svr_.predict(Z)

class ValidationWeightedMSKSVREnsemble(BaseEstimator, RegressorMixin):
    def __init__(self, n_members=8, base_params=None, tau=0.05, cv=3, seed=42):
        self.n_members = n_members
        self.base_params = base_params or {}
        self.tau = tau
        self.cv = cv
        self.seed = seed

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        rng = np.random.RandomState(self.seed)
        self.members_ = []
        self.validation_rmse_ = []
        kfold = KFold(n_splits=self.cv, shuffle=True, random_state=self.seed)
        base_params = self.base_params
        for member_id in range(self.n_members):
            C = base_params.get("C", 10.0) * (0.5 + 1.5 * rng.rand())
            epsilon = base_params.get("eps", 0.01) * (0.5 + 1.5 * rng.rand())
            base_gamma = base_params.get("base_g", 0.05) * (0.5 + 1.5 * rng.rand())
            n_scales = base_params.get("n_scales", 3)
            spread = base_params.get("spread", 1.6)
            gammas = tuple(base_gamma * (spread ** k) for k in range(n_scales))
            model = MultiScaleKernelSVR(gammas=gammas, C=C, epsilon=epsilon, seed=self.seed + member_id)
            errors = []
            for train_idx, val_idx in kfold.split(X):
                member = clone(model)
                member.fit(X[train_idx], y[train_idx])
                prediction = member.predict(X[val_idx])
                errors.append(np.sqrt(mean_squared_error(y[val_idx], prediction)))
            self.validation_rmse_.append(float(np.mean(errors)))
            final_member = clone(model)
            final_member.fit(X, y)
            self.members_.append(final_member)
        errors = np.asarray(self.validation_rmse_)
        logits = -(errors - errors.min()) / (self.tau + 1e-8)
        weights = np.exp(logits - logits.max())
        self.weights_ = weights / weights.sum()
        return self

    def predict(self, X):
        predictions = np.stack([member.predict(X) for member in self.members_], axis=0)
        return np.average(predictions, axis=0, weights=self.weights_)

def optimize_msk_svr(X_fit, y_fit, n_trials=20, cv=3, seed=42):
    def objective(trial):
        C = trial.suggest_float("C", 1.0, 200.0, log=True)
        epsilon = trial.suggest_float("eps", 1e-3, 0.2, log=True)
        base_gamma = trial.suggest_float("base_g", 1e-3, 1.0, log=True)
        n_scales = trial.suggest_int("n_scales", 2, 5)
        spread = trial.suggest_float("spread", 0.3, 3.0)
        gammas = tuple(base_gamma * (spread ** k) for k in range(n_scales))
        model = MultiScaleKernelSVR(gammas=gammas, C=C, epsilon=epsilon, seed=seed)
        kfold = KFold(n_splits=cv, shuffle=True, random_state=seed)
        errors = []
        for train_idx, val_idx in kfold.split(X_fit):
            current_model = clone(model)
            current_model.fit(X_fit[train_idx], y_fit[train_idx])
            prediction = current_model.predict(X_fit[val_idx])
            errors.append(np.sqrt(mean_squared_error(y_fit[val_idx], prediction)))
        return float(np.mean(errors))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False, catch=(Exception,))
    return study.best_params, study.best_value

class MSKSVRPipeline(BaseEstimator, RegressorMixin):
    def __init__(self, seed=42, n_trials=20, cache_tag=None):
        self.seed = seed
        self.n_trials = n_trials
        self.cache_tag = cache_tag

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        self.scaler_ = RobustScaler(quantile_range=(5.0, 95.0))
        X_scaled = self.scaler_.fit_transform(X)
        def tune():
            best_params, best_score = optimize_msk_svr(X_scaled, y, n_trials=self.n_trials, cv=3, seed=self.seed)
            print(f"  [MS-SVR tuning] {self.cache_tag}: CV-RMSE={best_score:.4f}")
            return best_params
        if self.cache_tag is None:
            best_params = tune()
        else:
            best_params = cached(self.cache_tag, tune)
        self.model_ = ValidationWeightedMSKSVREnsemble(n_members=8, base_params=best_params, tau=0.05, cv=3, seed=self.seed)
        self.model_.fit(X_scaled, y)
        return self

    def predict(self, X):
        X_scaled = self.scaler_.transform(np.asarray(X, dtype=float))
        return self.model_.predict(X_scaled)

class TemporalAttention(layers.Layer):
    def __init__(self, units=32, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.projection = layers.Dense(units, activation="tanh")
        self.score = layers.Dense(1)

    def call(self, inputs):
        projected = self.projection(inputs)
        attention = tf.nn.softmax(self.score(projected), axis=1)
        return tf.reduce_sum(inputs * attention, axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config

class BaseAttentionFusion(layers.Layer):
    def __init__(self, n_base, init_logits=None, **kwargs):
        super().__init__(**kwargs)
        self.n_base = n_base
        if init_logits is None:
            bias_initializer = "zeros"
        else:
            initial_logits = np.asarray(init_logits, dtype="float32")
            def bias_initializer(shape, dtype=None):
                return tf.constant(initial_logits, dtype=dtype or tf.float32)
        self.logit = layers.Dense(n_base, kernel_initializer="zeros", bias_initializer=bias_initializer)

    def attention(self, hidden):
        return tf.nn.softmax(self.logit(hidden), axis=-1)

    def call(self, inputs):
        hidden, base_predictions = inputs
        weights = self.attention(hidden)
        return tf.reduce_sum(weights * base_predictions, axis=-1, keepdims=True)

    def get_config(self):
        config = super().get_config()
        config.update({"n_base": self.n_base})
        return config

class PositionalEmbedding(layers.Layer):
    def __init__(self, length, **kwargs):
        super().__init__(**kwargs)
        self.length = length

    def build(self, input_shape):
        self.position = self.add_weight(
            name="position",
            shape=(self.length, input_shape[-1]),
            initializer="random_normal",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):
        return inputs + self.position[None, :, :]

    def get_config(self):
        config = super().get_config()
        config.update({"length": self.length})
        return config

def tcn_block(x, filters, kernel_size, dilation, dropout=0.1):
    convolution = layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation, activation="relu")(x)
    convolution = layers.LayerNormalization()(convolution)
    convolution = layers.SpatialDropout1D(dropout)(convolution)
    convolution = layers.Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation, activation="relu")(convolution)
    convolution = layers.LayerNormalization()(convolution)
    if x.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(x)
    else:
        shortcut = x
    return layers.Activation("relu")(layers.Add()([convolution, shortcut]))

def build_proposed_model(n_channels, n_base=N_BASE, attention_init=None, use_bigru=True, use_temporal_attention=True, use_base_attention=True, use_base_learners=True, gru_units=32):
    sequence_input = layers.Input(shape=(SEQ_LEN, n_channels), name="sequence_input")
    if use_base_learners:
        base_input = layers.Input(shape=(n_base,), name="base_predictions")
    else:
        base_input = None
    x = layers.AveragePooling1D(4, name="pooling")(sequence_input)
    if use_bigru:
        x = layers.Bidirectional(layers.GRU(gru_units, return_sequences=True), name="bigru")(x)
    else:
        x = layers.Activation("linear", name="bigru")(x)
    if use_temporal_attention:
        x = TemporalAttention(32, name="temporal_attention")(x)
    else:
        x = layers.GlobalAveragePooling1D(name="global_pooling")(x)
    if use_base_learners:
        x = layers.Concatenate()([x, base_input])
    x = layers.Dropout(0.2, name="head_dropout_1")(x)
    x = layers.Dense(64, activation="relu", name="head_dense_1")(x)
    x = layers.Dropout(0.2, name="head_dropout_2")(x)
    x = layers.Dense(32, activation="relu", name="head_dense_2")(x)
    if use_base_learners:
        residual = layers.Dense(1, kernel_initializer="zeros", name="residual_head")(x)
        if use_base_attention:
            fused = BaseAttentionFusion(n_base, init_logits=attention_init, name="base_attention")([x, base_input])
        else:
            if attention_init is None:
                initial_weights = np.ones(n_base) / n_base
            else:
                logits = np.asarray(attention_init, dtype=float)
                logits = logits - logits.max()
                initial_weights = np.exp(logits)
                initial_weights = initial_weights / initial_weights.sum()
            initial_weights = initial_weights.reshape(-1, 1).astype("float32")
            fused = layers.Dense(
                1, use_bias=False, kernel_constraint=tf.keras.constraints.NonNeg(),
                kernel_initializer=lambda shape, dtype=None: tf.constant(initial_weights, dtype=dtype or tf.float32),
                name="static_base_fusion",
            )(base_input)
        output = layers.Add(name="final_output")([fused, residual])
        return models.Model([sequence_input, base_input], output)
    output = layers.Dense(1, name="output_head")(x)
    return models.Model(sequence_input, output)

BASELINES_ML = ["Linear Regression", "SVR", "Random Forest", "XGBoost", "MS-XGBoost", "MS-SVR Ensemble"]
BASELINES_NN = ["RNN", "LSTM", "BiGRU", "BiGRU-AM", "LSTM-Attn", "TCN", "TCN-BiGRU", "TCN-BiGRU-AM", "Transformer"]
ALL_MODELS = BASELINES_ML + BASELINES_NN + [PROPOSED]

CLASSICAL_MODELS = {
    "Linear Regression": lambda seed: LinearRegression(),
    "SVR": lambda seed: SVR(C=10.0, epsilon=0.01, gamma="scale"),
    "Random Forest": lambda seed: RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1),
}

def build_baseline(name, n_channels=3):
    inputs = layers.Input(shape=(SEQ_LEN, n_channels))
    if name == "RNN":
        x = layers.AveragePooling1D(4)(inputs)
        x = layers.SimpleRNN(64)(x)
    elif name == "LSTM":
        x = layers.AveragePooling1D(4)(inputs)
        x = layers.LSTM(64, return_sequences=True)(x)
        x = layers.LSTM(32)(x)
    elif name == "BiGRU":
        x = layers.AveragePooling1D(4)(inputs)
        x = layers.Bidirectional(layers.GRU(32))(x)
    elif name == "BiGRU-AM":
        x = layers.AveragePooling1D(4)(inputs)
        x = layers.Bidirectional(layers.GRU(32, return_sequences=True))(x)
        x = TemporalAttention(32)(x)
    elif name == "LSTM-Attn":
        x = layers.AveragePooling1D(4)(inputs)
        x = layers.LSTM(64, return_sequences=True)(x)
        x = TemporalAttention(32)(x)
    elif name in ["TCN", "TCN-BiGRU", "TCN-BiGRU-AM"]:
        x = inputs
        for dilation in [1, 2, 4, 8, 16, 32]:
            x = tcn_block(x, 32, 3, dilation)
        if name == "TCN":
            x = layers.GlobalAveragePooling1D()(x)
        else:
            x = layers.AveragePooling1D(4)(x)
            if name == "TCN-BiGRU":
                x = layers.Bidirectional(layers.GRU(32))(x)
            else:
                x = layers.Bidirectional(layers.GRU(32, return_sequences=True))(x)
                x = TemporalAttention(32)(x)
    elif name == "Transformer":
        x = layers.AveragePooling1D(4)(inputs)
        x = layers.Dense(32)(x)
        x = PositionalEmbedding(SEQ_LEN // 4)(x)
        attention = layers.MultiHeadAttention(num_heads=4, key_dim=8)(x, x)
        x = layers.LayerNormalization()(layers.Add()([x, attention]))
        feedforward = layers.Dense(64, activation="relu")(x)
        feedforward = layers.Dense(32)(feedforward)
        x = layers.LayerNormalization()(layers.Add()([x, feedforward]))
        x = layers.GlobalAveragePooling1D()(x)
    else:
        raise ValueError(f"Unknown baseline: {name}")
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.1)(x)
    return models.Model(inputs, layers.Dense(1)(x))

def make_xgb(seed=42):
    return xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=7, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=seed, n_jobs=-1,
    )

def out_of_fold_stage(model_factory, X_fit, y_fit, other_sets, seed=42):
    oof_predictions = np.zeros(len(X_fit))
    splits = KFold(n_splits=5, shuffle=True, random_state=seed).split(X_fit)
    for train_idx, val_idx in splits:
        model = model_factory(seed)
        model.fit(X_fit[train_idx], y_fit[train_idx])
        oof_predictions[val_idx] = model.predict(X_fit[val_idx])
    final_model = model_factory(seed)
    final_model.fit(X_fit, y_fit)
    final_predictions = [final_model.predict(X).reshape(-1, 1) for X in other_sets]
    return final_model, oof_predictions.reshape(-1, 1), final_predictions

def multi_scale_base_stage(model_factory, feature_sets_fit, y_fit, feature_sets_others, seed=42):
    oof_blocks = []
    prediction_blocks = [[] for _ in feature_sets_others]
    for scale_index in range(len(feature_sets_fit)):
        X_fit = feature_sets_fit[scale_index]
        other_sets = [features[scale_index] for features in feature_sets_others]
        scale_factory = (lambda seed_value, i=scale_index: model_factory(seed_value, i))
        _, oof, predictions = out_of_fold_stage(scale_factory, X_fit, y_fit, other_sets, seed=seed)
        oof_blocks.append(oof)
        for set_index, prediction in enumerate(predictions):
            prediction_blocks[set_index].append(prediction)
    oof_matrix = np.hstack(oof_blocks)
    prediction_matrices = [np.hstack(predictions) for predictions in prediction_blocks]
    return oof_matrix, prediction_matrices

def nonnegative_linear_stack(predictions, y_fit, other_prediction_sets):
    meta_model = LinearRegression(positive=True)
    meta_model.fit(predictions, y_fit)
    fit_prediction = meta_model.predict(predictions).reshape(-1, 1)
    other_predictions = [meta_model.predict(prediction_set).reshape(-1, 1) for prediction_set in other_prediction_sets]
    return fit_prediction, other_predictions, meta_model

def fit_neural_model(model, X_sequence, X_base, y, validation=None, learning_rate=5e-4, epochs=200, batch_size=16, early_stopping_patience=25, reduce_lr_patience=10):
    metrics = [tf.keras.metrics.MeanAbsoluteError(name="mae")]
    try:
        metrics.append(tf.keras.metrics.R2Score(name="r2"))
        r2_available = True
    except AttributeError:
        r2_available = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss="mse", metrics=metrics)
    monitor = "val_loss" if validation is not None else "loss"
    model_callbacks = [
        callbacks.EarlyStopping(monitor=monitor, patience=early_stopping_patience, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor=monitor, patience=reduce_lr_patience, factor=0.5, min_lr=1e-7),
    ]
    train_inputs = ([X_sequence] if X_base is None else [X_sequence, X_base])
    validation_data = None
    if validation is not None:
        X_val_sequence, X_val_base, y_val = validation
        validation_data = ([X_val_sequence] if X_val_base is None else [X_val_sequence, X_val_base], y_val)
    history = model.fit(
        train_inputs, y, validation_data=validation_data, epochs=epochs,
        batch_size=batch_size, verbose=0, callbacks=model_callbacks,
    )
    history_dict = history.history
    if not r2_available:
        history_dict["val_r2"] = [np.nan] * len(history_dict["loss"])
    return history_dict

def neural_predict(model, X_sequence, X_base, y_scaler):
    inputs = ([X_sequence] if X_base is None else [X_sequence, X_base])
    prediction = model.predict(inputs, verbose=0).reshape(-1, 1)
    return y_scaler.inverse_transform(prediction).flatten()

def measure_inference_time(model, X_sequence, X_base, repetitions=5):
    inputs = ([X_sequence] if X_base is None else [X_sequence, X_base])
    model.predict(inputs, verbose=0)
    start = time.time()
    for _ in range(repetitions):
        model.predict(inputs, verbose=0)
    elapsed = time.time() - start
    return elapsed * 1000.0 / (repetitions * len(X_sequence))

def train_or_load_neural_model(tag, build_function, seed, X_sequence_train, X_base_train, y_train, validation=None, **fit_kwargs):
    weights_path = wpath(tag)
    metadata_path = os.path.join(CACHE_DIR, slug(tag) + ".json")
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    np.random.seed(seed)
    model = build_function()
    if os.path.exists(weights_path) and os.path.exists(metadata_path):
        model.load_weights(weights_path)
        with open(metadata_path, "r") as f:
            metadata = json.load(f)
        return model, metadata["training_time"], metadata["history"]
    start = time.time()
    history = fit_neural_model(model, X_sequence_train, X_base_train, y_train, validation=validation, **fit_kwargs)
    training_time = time.time() - start
    model.save_weights(weights_path)
    with open(metadata_path, "w") as f:
        json.dump({
            "training_time": training_time,
            "history": {k: [float(v) if v is not None else None for v in values] for k, values in history.items()},
        }, f)
    return model, training_time, history

def load_proposed_model(tag, n_channels=3, **model_kwargs):
    tf.keras.backend.clear_session()
    model = build_proposed_model(n_channels, **model_kwargs)
    model.load_weights(wpath(tag))
    return model

def plot_training_history(history, target, model_name, filename):
    if history is None or "loss" not in history:
        print(f"No training history available for {model_name}.")
        return
    epochs = np.arange(1, len(history["loss"]) + 1)
    has_validation = ("val_loss" in history and len(history["val_loss"]) > 0)
    has_mae = "mae" in history and len(history["mae"]) > 0
    has_val_mae = "val_mae" in history and len(history["val_mae"]) > 0
    has_val_r2 = ("val_r2" in history and len(history["val_r2"]) > 0 and not np.all(np.isnan(history["val_r2"])))
    n_plots = 2 if not has_val_r2 else 3
    fig, axes = plt.subplots(1, n_plots, figsize=(5.5 * n_plots, 5))
    axes[0].plot(epochs, history["loss"], "b-", linewidth=2, label="Training loss (MSE)")
    if has_validation:
        axes[0].plot(epochs, history["val_loss"], "r--", linewidth=2, label="Validation loss (MSE)")
        best_epoch = int(np.argmin(history["val_loss"])) + 1
    else:
        best_epoch = int(np.argmin(history["loss"])) + 1
    axes[0].axvline(best_epoch, color="green", linestyle=":", linewidth=1.5, label=f"Best epoch = {best_epoch}")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss (MSE)")
    axes[0].set_title(f"{model_name}\n{target}: Training and validation loss", fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.5)
    if has_mae:
        axes[1].plot(epochs, history["mae"], "b-", linewidth=2, label="Training MAE")
    if has_val_mae:
        axes[1].plot(epochs, history["val_mae"], "r--", linewidth=2, label="Validation MAE")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE")
    axes[1].set_title(f"{model_name}\n{target}: Training and validation MAE", fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)
    if has_val_r2:
        axes[2].plot(epochs, history["val_r2"], "g-", linewidth=2, label="Validation R²")
        axes[2].axhline(0, color="black", linestyle="--", linewidth=1)
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylabel("R² score")
        axes[2].set_title(f"{model_name}\n{target}: Validation R²", fontweight="bold")
        axes[2].legend()
        axes[2].grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(filename, dpi=DPI)
    plt.show()
    plt.close(fig)

def plot_all_model_training_curves(histories, target, filename, metric="loss", validation_metric="val_loss"):
    available_models = [m for m, h in histories.items() if h is not None and metric in h]
    if len(available_models) == 0:
        print(f"No histories available for {target}.")
        return
    fig = plt.figure(figsize=(13, 6))
    for model_name in available_models:
        history = histories[model_name]
        epochs = np.arange(1, len(history[metric]) + 1)
        plt.plot(epochs, history[metric], linewidth=2, label=f"{model_name} - training")
        if validation_metric in history:
            plt.plot(epochs, history[validation_metric], linestyle="--", linewidth=2, label=f"{model_name} - validation")
    plt.xlabel("Epoch")
    if metric == "loss":
        plt.ylabel("Loss (MSE)")
    elif metric == "mae":
        plt.ylabel("MAE")
    else:
        plt.ylabel(metric)
    plt.title(f"{target}: Training and validation curves of neural models", fontweight="bold")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(filename, dpi=DPI)
    plt.show()
    plt.close(fig)

def make_bundle(tag, train_idx, validation_idx, calibration_idx, test_idx, target, seed, channels=(0, 1, 2)):
    y_all = y_of(target)
    channels = list(channels)
    sig_source = b"".join(np.asarray(idx, dtype=np.int64).tobytes() for idx in (train_idx, validation_idx, calibration_idx, test_idx)) + f"|{target}|{channels}|{seed}".encode()
    sig = hashlib.md5(sig_source).hexdigest()[:8]
    sequence_scaler = RobustScaler(quantile_range=(5.0, 95.0))
    feature_scaler = RobustScaler(quantile_range=(5.0, 95.0))
    target_scaler = MinMaxScaler()
    sequence_scaler.fit(X_all[train_idx][:, :, channels].reshape(-1, len(channels)))
    feature_scaler.fit(extract_stat_features(X_all[train_idx], channels))
    target_scaler.fit(y_all[train_idx].reshape(-1, 1))
    def scale_sequence_set(indices):
        return scale_sequence(sequence_scaler, X_all[indices][:, :, channels])
    def scale_stat_features(indices):
        return feature_scaler.transform(extract_stat_features(X_all[indices], channels))
    def scale_target(indices):
        return target_scaler.transform(y_all[indices].reshape(-1, 1)).flatten()
    def multi_scale_features(indices):
        return [extract_ms_features(X_all[indices], channels, scale) for scale in XGB_SCALES]

    X_seq_train = scale_sequence_set(train_idx)
    X_seq_validation = scale_sequence_set(validation_idx)
    X_seq_calibration = scale_sequence_set(calibration_idx)
    X_seq_test = scale_sequence_set(test_idx)
    X_stat_train = scale_stat_features(train_idx)
    X_stat_validation = scale_stat_features(validation_idx)
    X_stat_calibration = scale_stat_features(calibration_idx)
    X_stat_test = scale_stat_features(test_idx)
    y_train = scale_target(train_idx)
    y_validation = scale_target(validation_idx)
    y_calibration = scale_target(calibration_idx)
    y_test = y_all[test_idx]

    def build_base_predictions():
        start_time = time.time()
        xgb_model, xgb_oof, xgb_predictions = out_of_fold_stage(
            lambda seed_value: make_xgb(seed_value), X_stat_train, y_train,
            [X_stat_validation, X_stat_calibration, X_stat_test], seed=seed,
        )
        ms_feature_train = multi_scale_features(train_idx)
        ms_feature_validation = multi_scale_features(validation_idx)
        ms_feature_calibration = multi_scale_features(calibration_idx)
        ms_feature_test = multi_scale_features(test_idx)
        ms_xgb_oof, ms_xgb_predictions = multi_scale_base_stage(
            lambda seed_value, scale_index: make_xgb(seed_value), ms_feature_train, y_train,
            [ms_feature_validation, ms_feature_calibration, ms_feature_test], seed=seed,
        )
        svr_factory = lambda seed_value, scale_index: MSKSVRPipeline(
            seed=seed_value, n_trials=20,
            cache_tag=f"{tag}_{sig}_svr_scale{XGB_SCALES[scale_index]}_s{seed_value}",
        )
        ms_svr_oof, ms_svr_predictions = multi_scale_base_stage(
            svr_factory, ms_feature_train, y_train,
            [ms_feature_validation, ms_feature_calibration, ms_feature_test], seed=seed,
        )
        _, _, ms_xgb_stack_meta = nonnegative_linear_stack(ms_xgb_oof, y_train, [ms_xgb_predictions[0], ms_xgb_predictions[1], ms_xgb_predictions[2]])
        _, _, ms_svr_stack_meta = nonnegative_linear_stack(ms_svr_oof, y_train, [ms_svr_predictions[0], ms_svr_predictions[1], ms_svr_predictions[2]])
        joint_oof = np.hstack([ms_xgb_oof, ms_svr_oof])
        joint_predictions = [np.hstack([x, s]) for x, s in zip(ms_xgb_predictions, ms_svr_predictions)]
        _, _, joint_stack_meta = nonnegative_linear_stack(joint_oof, y_train, joint_predictions)
        return {
            "training_time": time.time() - start_time, "xgb_oof": xgb_oof, "xgb_predictions": xgb_predictions,
            "ms_xgb_oof": ms_xgb_oof, "ms_xgb_predictions": ms_xgb_predictions,
            "ms_svr_oof": ms_svr_oof, "ms_svr_predictions": ms_svr_predictions,
            "joint_oof": joint_oof, "joint_predictions": joint_predictions,
            "xgb_stack_meta": ms_xgb_stack_meta, "svr_stack_meta": ms_svr_stack_meta,
            "joint_stack_meta": joint_stack_meta,
        }

    base_results = cached(f"bundle_{tag}_{sig}", build_base_predictions)

    return {
        "tag": tag, "sig": sig, "target": target, "channels": channels,
        "sequence_scaler": sequence_scaler, "feature_scaler": feature_scaler, "target_scaler": target_scaler,
        "X_seq_train": X_seq_train, "X_seq_validation": X_seq_validation, "X_seq_calibration": X_seq_calibration, "X_seq_test": X_seq_test,
        "X_stat_train": X_stat_train, "X_stat_validation": X_stat_validation, "X_stat_calibration": X_stat_calibration, "X_stat_test": X_stat_test,
        "y_train": y_train, "y_validation": y_validation, "y_calibration": y_calibration, "y_test": y_test,
        "test_indices": test_idx, "single_xgb_oof": base_results["xgb_oof"], "single_xgb_predictions": base_results["xgb_predictions"],
        "ms_xgb_oof": base_results["ms_xgb_oof"], "ms_xgb_predictions": base_results["ms_xgb_predictions"],
        "ms_svr_oof": base_results["ms_svr_oof"], "ms_svr_predictions": base_results["ms_svr_predictions"],
        "joint_oof": base_results["joint_oof"], "joint_predictions": base_results["joint_predictions"],
        "xgb_stack_meta": base_results["xgb_stack_meta"], "svr_stack_meta": base_results["svr_stack_meta"],
        "joint_stack_meta": base_results["joint_stack_meta"], "base_training_time": base_results["training_time"],
    }

def evaluate_model(name, bundle, seed, tag, model_kwargs=None):
    target_scaler = bundle["target_scaler"]
    n_channels = bundle["X_seq_train"].shape[-1]
    tag = neural_tag(bundle, tag)

    if name == "XGBoost":
        prediction = target_scaler.inverse_transform(bundle["single_xgb_predictions"][2]).flatten()
        return {"prediction": prediction, "training_time": bundle["base_training_time"], "parameters": np.nan, "inference_time": np.nan, "history": None}

    if name == "MS-XGBoost":
        prediction = target_scaler.inverse_transform(bundle["xgb_stack_meta"].predict(bundle["ms_xgb_predictions"][2]).reshape(-1, 1)).flatten()
        return {"prediction": prediction, "training_time": bundle["base_training_time"], "parameters": np.nan, "inference_time": np.nan, "history": None}

    if name == "MS-SVR Ensemble":
        prediction = target_scaler.inverse_transform(bundle["svr_stack_meta"].predict(bundle["ms_svr_predictions"][2]).reshape(-1, 1)).flatten()
        return {"prediction": prediction, "training_time": bundle["base_training_time"], "parameters": np.nan, "inference_time": np.nan, "history": None}

    if name in CLASSICAL_MODELS:
        model = CLASSICAL_MODELS[name](seed)
        start = time.time()
        model.fit(bundle["X_stat_train"], bundle["y_train"])
        training_time = time.time() - start
        scaled_prediction = model.predict(bundle["X_stat_test"]).reshape(-1, 1)
        prediction = target_scaler.inverse_transform(scaled_prediction).flatten()
        return {"prediction": prediction, "training_time": training_time, "parameters": np.nan, "inference_time": np.nan, "history": None}

    if name == PROPOSED:
        arch_kwargs = dict(model_kwargs or {})
        base_selection = arch_kwargs.pop("base_selection", "all")
        use_base_learners = arch_kwargs.pop("use_base_learners", True)
        fit_kwargs = {}
        n_scales = len(XGB_SCALES)
        selected_columns = {"all": list(range(2 * n_scales)), "xgb": list(range(n_scales)), "svr": list(range(n_scales, 2 * n_scales))}[base_selection]
        stack_weights = np.clip(bundle["joint_stack_meta"].coef_[selected_columns], 0, None)
        if stack_weights.sum() > 0:
            stack_weights = stack_weights / stack_weights.sum()
        else:
            stack_weights = np.ones(len(selected_columns)) / len(selected_columns)
        attention_init = np.log(stack_weights + 1e-2)
        build_function = lambda: build_proposed_model(
            n_channels, n_base=len(selected_columns), attention_init=attention_init,
            use_base_learners=use_base_learners, **arch_kwargs,
        )
        if use_base_learners:
            X_base_train = bundle["joint_oof"][:, selected_columns]
            X_base_validation = bundle["joint_predictions"][0][:, selected_columns]
            X_base_calibration = bundle["joint_predictions"][1][:, selected_columns]
            X_base_test = bundle["joint_predictions"][2][:, selected_columns]
        else:
            X_base_train = None
            X_base_validation = None
            X_base_calibration = None
            X_base_test = None
    else:
        build_function = lambda: build_baseline(name, n_channels)
        X_base_train = None
        X_base_validation = None
        X_base_calibration = None
        X_base_test = None
        fit_kwargs = {"learning_rate": 1e-3, "epochs": 150, "early_stopping_patience": 20, "reduce_lr_patience": 8}

    model, training_time, history = train_or_load_neural_model(
        tag, build_function, seed, bundle["X_seq_train"], X_base_train, bundle["y_train"],
        validation=(bundle["X_seq_validation"], X_base_validation, bundle["y_validation"]), **fit_kwargs,
    )
    prediction = neural_predict(model, bundle["X_seq_test"], X_base_test, target_scaler)
    inference_time = measure_inference_time(model, bundle["X_seq_test"], X_base_test)
    return {"prediction": prediction, "training_time": training_time, "parameters": model.count_params(), "inference_time": inference_time, "history": history}

def parity_plot(ax, y_true, y_pred, title):
    relative_error = (np.abs(y_pred - y_true) / np.maximum(np.abs(y_true), 1.0) * 100.0)
    scatter = ax.scatter(y_true, y_pred, c=relative_error, cmap="coolwarm", s=16, edgecolors="black", linewidth=0.2, vmin=0, vmax=max(np.percentile(relative_error, 95), 1e-6))
    lower = min(y_true.min(), y_pred.min())
    upper = max(y_true.max(), y_pred.max())
    ax.plot([lower, upper], [lower, upper], "r--", linewidth=1.5, label="Ideal (Y = X)")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", fontsize=8)
    plt.colorbar(scatter, ax=ax, label="Relative error (%)")

def metric_bar_plot(dataframe, models_to_plot, targets, filename):
    figure, axes = plt.subplots(len(targets), len(METRICS), figsize=(4.8 * len(METRICS), 4.8 * len(targets)), squeeze=False)
    for target_index, target in enumerate(targets):
        subset = dataframe[dataframe["Target"] == target].set_index("Model").loc[models_to_plot]
        for metric_index, metric in enumerate(METRICS):
            ax = axes[target_index, metric_index]
            colors = ["#d62728" if model == PROPOSED else "#7f7f7f" for model in subset.index]
            ax.bar(range(len(subset)), subset[metric].values, color=colors, edgecolor="black")
            ax.set_xticks(range(len(subset)))
            ax.set_xticklabels(subset.index, rotation=75, ha="right", fontsize=8)
            ax.set_title(f"{target}: {metric}", fontweight="bold")
            ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(out(filename), dpi=DPI)
    plt.show()
    plt.close(figure)

def parity_figure(split_result, label):
    figure, axes = plt.subplots(1, len(TARGETS), figsize=(7 * len(TARGETS), 6), squeeze=False)
    for target_index, target in enumerate(TARGETS):
        parity_plot(axes[0, target_index], split_result["results"][target]["y_test"], split_result["predictions"][(target, PROPOSED)], f"{target}: Proposed ({label})")
    plt.tight_layout()
    plt.savefig(out(f"Fig_Parity_{label}.png"), dpi=DPI)
    plt.show()
    plt.close(figure)

def run_split(mode, label, model_names):
    splits = {target: make_splits(target, mode) for target in TARGETS}
    results, rows, predictions, histories = {}, [], {}, {}
    for target in TARGETS:
        sp = splits[target]
        print(f"\n===== Target: {target} | train={len(sp['train'])} val={len(sp['val'])} cal={len(sp['cal'])} test={len(sp['test'])} =====")
        bundle = make_bundle(tag=f"{mode}_{target}", train_idx=sp["train"], validation_idx=sp["val"], calibration_idx=sp["cal"], test_idx=sp["test"], target=target, seed=SEED)
        results[target] = bundle
        print(f"  Joint stack weights: {np.round(bundle['joint_stack_meta'].coef_, 3)}")
        for model_name in model_names:
            result = evaluate_model(model_name, bundle, SEED, f"{mode}_{target}_{model_name}_s{SEED}")
            metrics = get_metrics(bundle["y_test"], result["prediction"])
            rows.append({"Model": model_name, "Target": target, **metrics, "Parameters": result["parameters"], "Training Time (s)": result["training_time"], "Inference (ms/sample)": result["inference_time"]})
            predictions[(target, model_name)] = result["prediction"]
            histories[(target, model_name)] = result["history"]
            print(f"  {model_name:45s} RMSE={metrics['RMSE']:.4f} MAE={metrics['MAE']:.4f} R2={metrics['R2']:.4f}")
            if PLOT_TRAINING_CURVES and result["history"] is not None and (PLOT_BASELINE_CURVES or model_name == PROPOSED):
                plot_training_history(result["history"], target, model_name, out(f"Fig_Training_History_{label}_{target}_{slug(model_name)}.png"))
    table = pd.DataFrame(rows)
    table.to_csv(out(f"Table2_{label}_Baseline_Comparison.csv"), index=False)
    table[table["Model"] == PROPOSED].drop(columns=["Model"]).set_index("Target").to_csv(out(f"Table1_{label}_Proposed_Performance.csv"))
    display(table.sort_values(by=["Target", "R2"], ascending=[True, False]))
    metric_bar_plot(table, model_names, TARGETS, f"Fig_{label}_Model_Comparison.png")
    split_result = {"splits": splits, "results": results, "predictions": predictions, "histories": histories, "table": table}
    parity_figure(split_result, label)
    if PLOT_ALL_MODEL_CURVES:
        neural_names = [m for m in BASELINES_NN + [PROPOSED] if m in model_names]
        for target in TARGETS:
            neural_model_histories = {model_name: histories[(target, model_name)] for model_name in neural_names}
            plot_all_model_training_curves(histories=neural_model_histories, target=target, filename=out(f"Fig_{label}_All_Neural_Training_Curves_{target}.png"), metric="loss", validation_metric="val_loss")
            plot_all_model_training_curves(histories=neural_model_histories, target=target, filename=out(f"Fig_{label}_All_Neural_MAE_Curves_{target}.png"), metric="mae", validation_metric="val_mae")
    return split_result

RANDOM = run_split("random", "RandomSplit", ALL_MODELS) if RUN_RANDOM_SPLIT else None
CHRONO = None
REF = RANDOM
REF_MODE = "random"

if REF is None:
    RUN_ABLATION = RUN_CHANNEL_ABLATION = RUN_ROBUSTNESS = False
    RUN_UNCERTAINTY = False

if RUN_ABLATION:
    print("\n" + "=" * 60)
    print("ABLATION STUDY")
    print("=" * 60)
    ablation_configs = {
        "Single-scale XGBoost": "XGB", "MS-XGBoost only": "MSX", "MS-SVR ensemble only": "MSV",
        "Static MS-XGB + MS-SVR stack": "STACK", "Encoder only (no base learners)": {"use_base_learners": False},
        "Encoder + MS-XGB bases": {"base_selection": "xgb"}, "Encoder + MS-SVR bases": {"base_selection": "svr"},
        "Static base fusion (no base attention)": {"use_base_attention": False}, "Without temporal attention": {"use_temporal_attention": False},
        "Without BiGRU (pooled seq -> temporal attention)": {"use_bigru": False}, "Full proposed model": {},
    }
    ablation_reference = {"XGB": "XGBoost", "MSX": "MS-XGBoost", "MSV": "MS-SVR Ensemble"}
    ablation_rows = []
    for target in TARGETS:
        bundle = REF["results"][target]
        for configuration_name, configuration in ablation_configs.items():
            if isinstance(configuration, str):
                if configuration == "STACK":
                    prediction = bundle["target_scaler"].inverse_transform(bundle["joint_stack_meta"].predict(bundle["joint_predictions"][2]).reshape(-1, 1)).flatten()
                else:
                    reference_model = ablation_reference[configuration]
                    prediction = REF["predictions"][(target, reference_model)]
            elif configuration == {}:
                prediction = REF["predictions"][(target, PROPOSED)]
            else:
                result = evaluate_model(PROPOSED, bundle, SEED, f"ablation_{target}_{configuration_name}_s{SEED}", model_kwargs=configuration)
                prediction = result["prediction"]
                if PLOT_TRAINING_CURVES:
                    plot_training_history(result["history"], target, configuration_name, out(f"Fig_Training_History_{target}_{slug(configuration_name)}.png"))
            metrics = get_metrics(bundle["y_test"], prediction)
            ablation_rows.append({"Configuration": configuration_name, "Target": target, **metrics})
    ablation_table = pd.DataFrame(ablation_rows)
    ablation_table.to_csv(out("Table4_Ablation_Study.csv"), index=False)
    display(ablation_table)

if RUN_CHANNEL_ABLATION:
    print("\n" + "=" * 60)
    print("SENSOR-CHANNEL ABLATION")
    print("=" * 60)
    channel_sets = {"Voltage only": (0,), "Voltage + Temperature": (0, 2), "Voltage + Current": (0, 1), "Voltage + Current + Temperature": (0, 1, 2)}
    channel_rows = []
    for target in TARGETS:
        sp = REF["splits"][target]
        for channel_name, channels in channel_sets.items():
            if channels == (0, 1, 2):
                prediction = REF["predictions"][(target, PROPOSED)]
                y_true = REF["results"][target]["y_test"]
            else:
                channel_bundle = make_bundle(tag=f"channels_{target}_{''.join(map(str, channels))}", train_idx=sp["train"], validation_idx=sp["val"], calibration_idx=sp["cal"], test_idx=sp["test"], target=target, seed=SEED, channels=channels)
                result = evaluate_model(PROPOSED, channel_bundle, SEED, f"channels_{target}_{''.join(map(str, channels))}_s{SEED}")
                prediction = result["prediction"]
                y_true = channel_bundle["y_test"]
                if PLOT_TRAINING_CURVES:
                    plot_training_history(result["history"], target, channel_name, out(f"Fig_Training_History_{target}_{slug(channel_name)}.png"))
            metrics = get_metrics(y_true, prediction)
            channel_rows.append({"Channels": channel_name, "Target": target, **metrics})
    channel_table = pd.DataFrame(channel_rows)
    channel_table.to_csv(out("Table4b_Channel_Ablation.csv"), index=False)
    display(channel_table)

if RUN_ROBUSTNESS:
    print("\n" + "=" * 60)
    print("NEURAL FUSION INITIALIZATION ROBUSTNESS")
    print("=" * 60)
    robustness_rows = []
    for target in TARGETS:
        bundle = REF["results"][target]
        for seed_value in ROBUST_SEEDS:
            result = evaluate_model(PROPOSED, bundle, seed_value, f"robustness_{target}_{PROPOSED}_s{seed_value}")
            metrics = get_metrics(bundle["y_test"], result["prediction"])
            robustness_rows.append({"Target": target, "Seed": seed_value, **metrics})
            if PLOT_TRAINING_CURVES and seed_value == SEED:
                plot_training_history(result["history"], target, f"{PROPOSED} (seed {seed_value})", out(f"Fig_Training_History_{target}_Robustness_Seed{seed_value}.png"))
    robustness_table = pd.DataFrame(robustness_rows)
    robustness_table.to_csv(out("Table5_NeuralFusion_Seed_Robustness.csv"), index=False)
    for target in TARGETS:
        subset = robustness_table[robustness_table["Target"] == target]
        print(f"{target}: RMSE {subset['RMSE'].mean():.4f} ± {subset['RMSE'].std():.4f}, MAE {subset['MAE'].mean():.4f} ± {subset['MAE'].std():.4f}, R2 {subset['R2'].mean():.4f} ± {subset['R2'].std():.4f}")

if RUN_UNCERTAINTY:
    print("\n" + "=" * 60)
    print("UNCERTAINTY QUANTIFICATION")
    print("=" * 60)
    conformal_rows = []
    figure, axes = plt.subplots(1, len(TARGETS), figsize=(14, 5), squeeze=False)
    axes = axes[0]
    for target_index, target in enumerate(TARGETS):
        bundle = REF["results"][target]
        target_scaler = bundle["target_scaler"]
        model = load_proposed_model(neural_tag(bundle, f"{REF_MODE}_{target}_{PROPOSED}_s{SEED}"), n_channels=bundle["X_seq_train"].shape[-1])
        calibration_true = target_scaler.inverse_transform(bundle["y_calibration"].reshape(-1, 1)).flatten()
        calibration_prediction = neural_predict(model, bundle["X_seq_calibration"], bundle["joint_predictions"][1], target_scaler)
        test_true = bundle["y_test"]
        test_prediction = neural_predict(model, bundle["X_seq_test"], bundle["joint_predictions"][2], target_scaler)
        conformity_scores = np.abs(calibration_prediction - calibration_true)
        n_calibration = len(conformity_scores)
        row = {"Target": target, "N_calibration": n_calibration}
        for coverage_level in [0.5, 0.8, 0.9, 0.95]:
            alpha = 1.0 - coverage_level
            quantile_level = min((1.0 - alpha) * (1.0 + 1.0 / n_calibration), 1.0)
            radius = np.quantile(conformity_scores, quantile_level, method="higher")
            lower_bound = test_prediction - radius
            upper_bound = test_prediction + radius
            coverage = np.mean((test_true >= lower_bound) & (test_true <= upper_bound)) * 100.0
            width = np.mean(upper_bound - lower_bound)
            row[f"Coverage@{int(coverage_level * 100)} (%)"] = coverage
            row[f"Width@{int(coverage_level * 100)}"] = width
        conformal_rows.append(row)
        radius_95 = np.quantile(conformity_scores, min(0.95 * (1.0 + 1.0 / n_calibration), 1.0), method="higher")
        lower_95 = test_prediction - radius_95
        upper_95 = test_prediction + radius_95
        order = np.argsort(test_true)
        ax = axes[target_index]
        ax.plot(test_true[order], test_prediction[order], "r-", linewidth=2, label="Prediction")
        ax.fill_between(test_true[order], lower_95[order], upper_95[order], color="green", alpha=0.3, label="95% conformal interval")
        ax.scatter(test_true[order], test_prediction[order], c="blue", s=15, alpha=0.6, edgecolors="white", linewidth=0.3, zorder=3)
        ax.set_title(f"{target}: Split conformal prediction", fontweight="bold")
        ax.set_xlabel(f"Actual {target}")
        ax.set_ylabel(f"Predicted {target}")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(out("Fig_SplitConformal_Uncertainty.png"), dpi=DPI)
    plt.show()
    plt.close(figure)
    conformal_table = pd.DataFrame(conformal_rows)
    conformal_table.to_csv(out("Table7_SplitConformal_Uncertainty.csv"), index=False)
    display(conformal_table)

    ensemble_rows = []
    figure, axes = plt.subplots(1, len(TARGETS), figsize=(14, 5), squeeze=False)
    axes = axes[0]
    for target_index, target in enumerate(TARGETS):
        bundle = REF["results"][target]
        target_scaler = bundle["target_scaler"]
        predictions = []
        for seed_value in ROBUST_SEEDS:
            try:
                model = load_proposed_model(neural_tag(bundle, f"robustness_{target}_{PROPOSED}_s{seed_value}"), n_channels=bundle["X_seq_train"].shape[-1])
                prediction = neural_predict(model, bundle["X_seq_test"], bundle["joint_predictions"][2], target_scaler)
                predictions.append(prediction)
            except Exception as error:
                print(f"Seed {seed_value} unavailable: {error}")
        if len(predictions) == 0:
            continue
        predictions = np.asarray(predictions)
        mean_prediction = predictions.mean(axis=0)
        std_prediction = predictions.std(axis=0)
        y_true = bundle["y_test"]
        order = np.argsort(y_true)
        ax = axes[target_index]
        ax.plot(y_true[order], mean_prediction[order], "r-", linewidth=2, label=f"Mean prediction ({len(predictions)} seeds)")
        ax.fill_between(y_true[order], mean_prediction[order] - 1.96 * std_prediction[order], mean_prediction[order] + 1.96 * std_prediction[order], color="gray", alpha=0.4, label="Approximate 95% ensemble band")
        ax.scatter(y_true[order], mean_prediction[order], c="blue", s=15, alpha=0.6, edgecolors="white", linewidth=0.3, zorder=3)
        ax.set_title(f"{target}: Neural initialization ensemble", fontweight="bold")
        ax.set_xlabel(f"Actual {target}")
        ax.set_ylabel(f"Predicted {target}")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)
        row = {"Target": target, "N_seeds": len(predictions), "Mean predictive std": std_prediction.mean(), "Mean 95% ensemble width": (2.0 * 1.96 * std_prediction).mean()}
        for coverage_level in [0.5, 0.8, 0.9, 0.95]:
            z_value = stats.norm.ppf(0.5 + coverage_level / 2.0)
            coverage = np.mean(np.abs(y_true - mean_prediction) <= z_value * std_prediction) * 100.0
            row[f"Coverage@{int(coverage_level * 100)} (%)"] = coverage
        ensemble_rows.append(row)
    plt.tight_layout()
    plt.savefig(out("Fig_NeuralEnsemble_Uncertainty.png"), dpi=DPI)
    plt.show()
    plt.close(figure)
    ensemble_table = pd.DataFrame(ensemble_rows)
    ensemble_table.to_csv(out("Table7b_NeuralEnsemble_Uncertainty.csv"), index=False)
    display(ensemble_table)

if REF is not None:
    print("\n" + "=" * 60)
    print("JOINT STACKING WEIGHTS")
    print("=" * 60)
    stacking_rows = []
    for target in TARGETS:
        weights = REF["results"][target]["joint_stack_meta"].coef_
        stacking_rows.append({"Target": target, **{f"Weight {name}": weights[index] for index, name in enumerate(BASE_NAMES)}})
    stacking_table = pd.DataFrame(stacking_rows)
    stacking_table.to_csv(out("Table12_Joint_Stacking_Weights.csv"), index=False)
    display(stacking_table)

    efficiency_table = REF["table"][["Model", "Target", "Parameters", "Training Time (s)", "Inference (ms/sample)"]]
    efficiency_table.to_csv(out("Table9_Computational_Efficiency.csv"), index=False)
    display(efficiency_table)

def banner(title):
    print("\n" + "-" * 3 + f" {title} " + "-" * 3)

def pretty(df):
    with pd.option_context("display.max_columns", None, "display.width", 250, "display.float_format", "{:.4f}".format):
        display(df)

REPORT_RENAME = {"Parameters": "Params", "Training Time (s)": "Train Time (s)"}
REPORT_COLS = (["Model", "Target"] + METRICS + ["Params", "Train Time (s)", "Inference (ms/sample)"])

def plot_test_predictions(split_result, label):
    fig, axes = plt.subplots(1, len(TARGETS), figsize=(7 * len(TARGETS), 5), squeeze=False)
    for i, target in enumerate(TARGETS):
        bundle = split_result["results"][target]
        order = np.argsort(bundle["test_indices"])
        y_true = bundle["y_test"][order]
        y_pred = split_result["predictions"][(target, PROPOSED)][order]
        ax = axes[0, i]
        ax.plot(y_true, "b-", linewidth=2, alpha=0.8, label=f"True {target}")
        ax.plot(y_pred, "r--", linewidth=2, label="Predicted (Proposed)")
        ax.set_title(f"{target} Prediction on Test Data", fontweight="bold")
        ax.set_xlabel("Test Sample Index")
        ax.set_ylabel(target)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend()
    plt.tight_layout()
    plt.savefig(out(f"Fig_TestPrediction_{label}.png"), dpi=DPI)
    plt.show()
    plt.close(fig)

def report_split(split_result, label):
    table = split_result["table"].rename(columns=REPORT_RENAME)[REPORT_COLS]
    banner(f"1. Comparative Performance base vs proposed [{label}]")
    pretty(table.sort_values(["Target", "R2"], ascending=[True, False]).reset_index(drop=True))
    banner(f"2. Proposed Model Performance [{label}]")
    pretty(table[table["Model"] == PROPOSED].drop(columns=["Model"]).set_index("Target"))
    plot_test_predictions(split_result, label)

if RANDOM is not None:
    report_split(RANDOM, "RandomSplit")

if RUN_ABLATION and "ablation_table" in globals():
    banner("3. ABLATION STUDY")
    pretty(ablation_table)

if RUN_CHANNEL_ABLATION and "channel_table" in globals():
    banner("4. Channel ablation")
    pretty(channel_table)

if RUN_ROBUSTNESS and "robustness_table" in globals():
    banner("5. ROBUSTNESS (5 Seeds)")
    pretty(robustness_table)
    print("\nMean / Std over seeds:")
    pretty(robustness_table.groupby("Target")[METRICS].agg(["mean", "std"]))

if RUN_UNCERTAINTY and "conformal_table" in globals():
    banner("6. Uncertainty Comparison")
    levels = [50, 80, 90, 95]
    comparison_rows = []
    for target in TARGETS:
        c = conformal_table[conformal_table["Target"] == target]
        if len(c):
            c = c.iloc[0]
            comparison_rows.append({
                "Target": target, "Method": "Split conformal",
                **{f"Coverage@{l} (%)": c[f"Coverage@{l} (%)"] for l in levels},
                "Mean width @95": c["Width@95"],
            })
        if "ensemble_table" in globals() and len(ensemble_table):
            e = ensemble_table[ensemble_table["Target"] == target]
            if len(e):
                e = e.iloc[0]
                comparison_rows.append({
                    "Target": target, "Method": "Neural init. ensemble",
                    **{f"Coverage@{l} (%)": e[f"Coverage@{l} (%)"] for l in levels},
                    "Mean width @95": e["Mean 95% ensemble width"],
                })
    uncertainty_comparison = pd.DataFrame(comparison_rows)
    uncertainty_comparison.to_csv(out("Table7c_Uncertainty_Comparison.csv"), index=False)
    pretty(uncertainty_comparison)

print("\nAll outputs saved in:")
print(OUTPUT_DIR)
print(sorted(filename for filename in os.listdir(OUTPUT_DIR) if filename.endswith((".csv", ".png"))))